# DecodeBound on Kaggle (free T4 GPU)

Produces fully **certified** `results/` for DecodeBound on a free Kaggle GPU.

**Before running:** open the notebook **Settings** panel (right sidebar) and set:
- **Accelerator** → `GPU T4 x2` (or `GPU P100`)
- **Internet** → `On`  (needed to clone the repo, install vLLM, and download the model)

**Then use `Save Version → Save & Run All (Commit)`** — the certified sweep
(2000 requests/point) takes ~4–5 h on a T4, so let it run in the background and
collect `decodebound_results.zip` from the version's Output tab when it finishes.
For a ~20 min interactive smoke test, set `--n-requests 128` in the sweep cell
and just Run All.

Notes:
- Uses **Qwen2.5-1.5B-Instruct** — small enough to fit a 16 GB T4 with room for KV cache.
  Bump to a 7B model only on an A100/L4 (a T4 can't hold it comfortably).
- Forces `--dtype half`: **T4 has no bf16**, and Qwen defaults to bf16.

In [ ]:
# 0. Confirm we actually got a GPU (DecodeBound refuses to run on CPU).
!nvidia-smi

In [ ]:
# 1. Clone the repo into the writable working dir.
%cd /kaggle/working
!rm -rf decodebound
!git clone --depth 1 https://github.com/wuisabel-gif/decodebound.git
%cd /kaggle/working/decodebound

In [ ]:
# 2. Install the analysis layer + the vLLM serving backend.
#    vLLM is large and may reinstall torch to match its pin — this cell is the slow one.
%pip install -q -e .
%pip install -q vllm

In [ ]:
# 3. Run the sweep. 2000 requests/point is what `certify` says a trustworthy p99
#    needs (~20 effective tail samples). Takes ~4-5h on a T4, so launch it with
#    'Save Version -> Save & Run All' and let it run in the background.
#    For a quick ~20min smoke test instead, drop --n-requests to 128.
#
#    T4 stability: the T4 (compute cap 7.5) can't use FlashAttention 2, and the
#    auto-fallback + CUDA-graph path crashes the engine over long runs. Force the
#    mature xformers backend and pass --enforce-eager to skip graph capture.
import os
os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

!python -m harness.cli check-gpu
!python -m harness.cli sweep \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --concurrency 1,2,4,8,16 \
    --n-requests 2000 \
    --prompt-len 512 --output-len 256 \
    --dtype half --max-model-len 4096 \
    --enforce-eager \
    --raw results/raw --yes

In [ ]:
# 4. Derive the figures and the sweep table from the raw per-request data.
!python -m harness.cli plot --raw results/raw --figures results/figures
!python -m harness.cli analyze --raw results/raw

In [ ]:
# 4b. Certify: should these numbers be believed? (exit 1 just means "not all TRUSTED")
!python -m harness.cli certify --raw results/raw || true

In [ ]:
# 5. Show the three figures inline.
from IPython.display import Image, display
from pathlib import Path
for name in ["prefill_decode.png", "pareto.png", "tail_latency.png"]:
    p = Path("results/figures") / name
    if p.exists():
        print(name)
        display(Image(str(p)))
    else:
        print("missing:", p)

In [ ]:
# 6. Bundle results/ so you can download it (Kaggle: Output tab) and commit it to the repo.
import shutil
shutil.make_archive("/kaggle/working/decodebound_results", "zip", "results")
print("Wrote /kaggle/working/decodebound_results.zip")
!cat results/raw/run_meta.json